Cell 1 – Install / Import

In [ ]:
# 📦 Telco Churn Prediction — Diamond Ultimate Elite Edition 💎
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

from datetime import datetime
from pathlib import Path

# === Config ===
DATA_PATH = "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
MODEL_DIR = "../backend/models/telco/"
MODEL_NAME = "sklearn_model.pkl"
FEATURES_NAME = "feature_names_sklearn.json"

# Ensure model directory exists
os.makedirs(MODEL_DIR, exist_ok=True)

# === 1. Load Data ===
df = pd.read_csv(DATA_PATH)
print(f"✅ Loaded dataset with shape: {df.shape}")

# === 2. Clean & Preprocess ===
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()

# Convert target
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Drop ID
df = df.drop(columns=['customerID'])

# Separate features and label
y = df['Churn']
X = df.drop(columns=['Churn'])

# One-hot encode categoricals
X = pd.get_dummies(X)

# Optional: Scale numerical features
scaler = StandardScaler()
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
X[num_cols] = scaler.fit_transform(X[num_cols])

# === 3. Split Data ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"🧪 Training on {X_train.shape[0]} samples, Testing on {X_test.shape[0]} samples")

# === 4. Train Model ===
model = LogisticRegression(max_iter=1000, solver='lbfgs')
model.fit(X_train, y_train)

# === 5. Evaluate ===
preds = model.predict(X_test)
report = classification_report(y_test, preds, digits=4)
conf_matrix = confusion_matrix(y_test, preds)

print("📊 Classification Report:\n", report)
print("🧩 Confusion Matrix:\n", conf_matrix)

# === 6. Save Model & Features ===
joblib.dump(model, os.path.join(MODEL_DIR, MODEL_NAME))
X.columns.to_series().to_json(os.path.join(MODEL_DIR, FEATURES_NAME), indent=2)

print(f"✅ Model saved to: {MODEL_DIR}{MODEL_NAME}")
print(f"🧠 Feature list saved to: {MODEL_DIR}{FEATURES_NAME}")
print("🏁 Done at", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))